<a href="https://colab.research.google.com/github/davideliseobesarezchavez-sketch/David/blob/main/Copia_de_Copia_de_Te_damos_la_bienvenida_a_Colaboratory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# SCRIPT COMPLETO: EXPERIMENTOS DE REDES NEURONALES Y REGRESIÓN LINEAL
# ==============================================================================

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def print_separador(texto):
    print("\n" + "="*70)
    print(f"  {texto}")
    print("="*70)

In [ ]:
# ==============================================================================
# CONFIGURACIÓN INICIAL: CARGA DE DATOS (FASHION MNIST)
# ==============================================================================
print_separador("CONFIGURACIÓN INICIAL Y PREPROCESAMIENTO DE DATOS")

# Cargar el set de datos de ropa correspondiente a tu imagen
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalizar los píxeles para que estén en el rango [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# Etiquetas correctas basadas en Fashion MNIST
nombres_clases = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("✓ Datos de Fashion MNIST cargados correctamente.")
print(f"✓ Formato de entrenamiento: {x_train.shape}")
print(f"✓ Formato de prueba: {x_test.shape}")

In [ ]:
# ==============================================================================
# PASO 1: INCREMENTAR ÉPOCAS DE ENTRENAMIENTO (10 VS 50)
# ==============================================================================
print_separador("PASO 1: COMPARATIVA DE ÉPOCAS (10 vs 50)")

def crear_modelo_base(neuronas=128, dos_capas=False):
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(neuronas, activation='relu')
    ])
    if dos_capas:
        model.add(layers.Dense(neuronas, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Entrenando modelo base durante 10 épocas...")
modelo_10 = crear_modelo_base()
modelo_10.fit(x_train, y_train, epochs=10, verbose=0)
_, acc_10 = modelo_10.evaluate(x_test, y_test, verbose=0)

print("Entrenando modelo base durante 50 épocas (Por favor, espera)...")
modelo_50 = crear_modelo_base()
modelo_50.fit(x_train, y_train, epochs=50, verbose=0)
_, acc_50 = modelo_50.evaluate(x_test, y_test, verbose=0)

print(f"\n[RESULTADOS EXPERIMENTO 1]")
print(f"-> Precisión obtenida con 10 épocas: {acc_10:.4f}")
print(f"-> Precisión obtenida con 50 épocas: {acc_50:.4f}")

In [ ]:
# ==============================================================================
# PASO 2: MODIFICAR CANTIDAD DE NEURONAS OCULTAS (64, 128, 256)
# ==============================================================================
print_separador("PASO 2: EVALUACIÓN DE NEURONAS EN CAPA OCULTA")

configuraciones = [64, 128, 256]
resultados_neuronas = {}

for n in configuraciones:
    print(f"Entrenando modelo con {n} neuronas en la capa oculta (10 épocas)...")
    modelo_n = crear_modelo_base(neuronas=n)
    modelo_n.fit(x_train, y_train, epochs=10, verbose=0)
    _, acc_n = modelo_n.evaluate(x_test, y_test, verbose=0)
    resultados_neuronas[n] = acc_n

print("\n[RESULTADOS EXPERIMENTO 2]")
mejor_n = 64
for n, acc in resultados_neuronas.items():
    print(f"-> Configuración {n} neuronas: Precisión = {acc:.4f}")
    if acc > resultados_neuronas[mejor_n]:
        mejor_n = n

print(f"\n>> Conclusión: La configuración con {mejor_n} neuronas ofrece los mejores resultados.")

In [ ]:
# ==============================================================================
# PASO 3: AGREGAR UNA SEGUNDA CAPA OCULTA
# ==============================================================================
print_separador("PASO 3: EVALUACIÓN DE UNA SEGUNDA CAPA OCULTA (PROFUNDIDAD)")

print("Entrenando modelo con dos capas ocultas (128 neuronas cada una, 10 épocas)...")
modelo_2capas = crear_modelo_base(neuronas=128, dos_capas=True)
modelo_2capas.fit(x_train, y_train, epochs=10, verbose=0)
_, acc_2capas = modelo_2capas.evaluate(x_test, y_test, verbose=0)

print("\n[RESULTADOS EXPERIMENTO 3]")
print(f"-> Precisión con 1 Capa Oculta (128 neuronas): {acc_10:.4f}")
print(f"-> Precisión con 2 Capas Ocultas (128 c/u):      {acc_2capas:.4f}")

In [ ]:
# ==============================================================================
# PASO 4: MOSTRAR PREDICCIONES DE 5 IMÁGENES DE PRUEBA
# ==============================================================================
print_separador("PASO 4: VISUALIZACIÓN DE PREDICCIONES (REAL VS PREDICCIÓN)")

# Tomar las primeras 5 imágenes del conjunto de prueba y predecir su categoría
predicciones = modelo_2capas.predict(x_test[:5], verbose=0)

plt.figure(figsize=(14, 3.5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_test[i], cmap='gray')

    clase_real = nombres_clases[y_test[i]]
    clase_pred = nombres_clases[np.argmax(predicciones[i])]

    # Asignar color verde si acertó, rojo si falló
    color_titulo = 'green' if clase_real == clase_pred else 'red'

    plt.title(f"Real: {clase_real}\nPred: {clase_pred}", color=color_titulo, fontsize=11, weight='bold')
    plt.axis('off')

plt.suptitle("Predicciones de Muestra (Modelo de 2 Capas)", fontsize=14, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# PASO 5: REGRESIÓN LINEAL CON NUEVOS DATOS
# ==============================================================================
print_separador("PASO 5: EJERCICIO DE REGRESIÓN LINEAL (TEMPERATURA VS VENTAS)")

# 1. Conjunto de datos base inicial
X_base = np.array([22, 25, 28, 30, 32, 35, 24, 27, 29, 31]).reshape(-1, 1)
y_base = np.array([150, 200, 250, 280, 310, 400, 180, 220, 240, 300])

reg_base = LinearRegression().fit(X_base, y_base)
r2_base = r2_score(y_base, reg_base.predict(X_base))

# 2. Agregar nuevos datos extremos (ej. días muy fríos o muy calurosos)
X_nuevos = np.array([14, 17, 38, 41]).reshape(-1, 1)
y_nuevos = np.array([65, 100, 490, 560])

# Combinar arrays antiguos con los nuevos datos
X_completo = np.vstack((X_base, X_nuevos))
y_completo = np.hstack((y_base, y_nuevos))

reg_completo = LinearRegression().fit(X_completo, y_completo)
r2_completo = r2_score(y_completo, reg_completo.predict(X_completo))

# 3. Mostrar métricas comparativas por consola
print("[ANÁLISIS DE MÉTRICAS]")
print(f"-> Modelo Inicial   | Coeficiente R²: {r2_base:.4f} | Pendiente: {reg_base.coef_[0]:.2f}")
print(f"-> Modelo Ampliado  | Coeficiente R²: {r2_completo:.4f} | Pendiente: {reg_completo.coef_[0]:.2f}")

# 4. Graficar comparativa visual de las líneas de regresión
plt.figure(figsize=(10, 6))
plt.scatter(X_base, y_base, color='blue', s=60, label='Datos Base Iniciales', zorder=3)
plt.scatter(X_nuevos, y_nuevos, color='red', marker='s', s=80, label='Nuevos Datos Agregados', zorder=3)

# Generar puntos ordenados para trazar las líneas rectas continuas
linea_x = np.linspace(10, 45, 100).reshape(-1, 1)
plt.plot(linea_x, reg_base.predict(linea_x), color='blue', linestyle='--', linewidth=2, label=f'Recta Inicial (R²={r2_base:.2f})')
plt.plot(linea_x, reg_completo.predict(linea_x), color='red', linestyle='-', linewidth=2, label=f'Recta Nueva (R²={r2_completo:.2f})')

plt.xlabel('Temperatura (°C)', fontsize=11)
plt.ylabel('Ventas ($)', fontsize=11)
plt.title('Impacto de Nuevos Datos en la Línea de Regresión', fontsize=13, weight='bold')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("\n" + "="*70)
print("  ¡Práctica completada con éxito!")
print("="*70)